# DiffMatte ONNX Runtime GPU: retriever サンプル可視化

DiffMatte 公式の `retriever_rgb.png` と `retriever_trimap.png` を、GTX 1070 向け ONNX Runtime GPU 経路で推論し、trimap・alpha・合成プレビューを可視化します。

事前にリポジトリ root で `uv sync --extra runtime-gpu --extra notebook --extra dev` を実行してください。GPU 経路は CUDA 11.8 / cuDNN 8 / ONNX Runtime GPU 1.16.3 を使用し、古い ORT の FusedConv 最適化は runtime 側で無効化しています。

In [ ]:
from __future__ import annotations

import hashlib
import json
import sys
import time
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'pyproject.toml').is_file():
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / 'pyproject.toml').is_file():
    raise RuntimeError('Run this notebook from the diffmatte_onnx repository.')

sys.path.insert(0, str(REPO_ROOT / 'src'))
ARTIFACT_DIR = REPO_ROOT / 'artifacts' / 'notebooks'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = REPO_ROOT / 'models' / 'diffmatte_vits_1024_ddim10_643x960_gtx1070.onnx'
MODEL_URL = ('https://github.com/yuki-inaho/diffmatte_onnx/releases/download/'
             'gtx1070-onnx-v0.1.1/diffmatte_vits_1024_ddim10_643x960_gtx1070.onnx')
MODEL_SHA256 = '3eebe380102041c7bd9da7c491f56ac4f30a6df76921de5a4a4e12ce5fe362b6'
IMAGE_PATH = REPO_ROOT / 'demo' / 'retriever_rgb.png'
TRIMAP_PATH = REPO_ROOT / 'demo' / 'retriever_trimap.png'
print(REPO_ROOT)

In [ ]:
def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as source:
        for block in iter(lambda: source.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

if not MODEL_PATH.is_file():
    MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
    print(f'Downloading {MODEL_URL}')
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)

actual_hash = sha256(MODEL_PATH)
if actual_hash != MODEL_SHA256:
    raise RuntimeError(f'Unexpected model SHA-256: {actual_hash}')
print({'model': str(MODEL_PATH), 'sha256': actual_hash, 'bytes': MODEL_PATH.stat().st_size})

In [ ]:
import onnxruntime as ort

from diffmatte_onnx.runtime import (
    load_inputs,
    make_noise,
    preload_cuda_dependencies,
    run_inference,
)

loaded_libraries = preload_cuda_dependencies()
available = ort.get_available_providers()
print({'onnxruntime': ort.__version__, 'available_providers': available})
assert 'CUDAExecutionProvider' in available, 'Install the runtime-gpu extra.'
print('preloaded:', *loaded_libraries, sep='\n  ')

In [ ]:
image, trimap = load_inputs(IMAGE_PATH, TRIMAP_PATH)
noise = make_noise(image, seed=0, noise_path=None)
print({'image': image.shape, 'trimap': trimap.shape, 'noise': noise.shape,
       'trimap_values': np.unique(trimap).tolist()})

In [ ]:
started = time.perf_counter()
alpha, providers = run_inference(MODEL_PATH, image, trimap, noise, provider='cuda')
elapsed_seconds = time.perf_counter() - started
assert 'CUDAExecutionProvider' in providers, providers
alpha_2d = np.clip(alpha[0, 0], 0.0, 1.0)
alpha_path = ARTIFACT_DIR / 'retriever_gpu_alpha.png'
Image.fromarray(np.rint(alpha_2d * 255.0).astype(np.uint8)).save(alpha_path)
print({'providers': providers, 'seconds': elapsed_seconds,
       'alpha_min': float(alpha_2d.min()), 'alpha_max': float(alpha_2d.max()),
       'alpha_png': str(alpha_path)})

In [ ]:
rgb = np.transpose(image[0], (1, 2, 0))
tri = trimap[0, 0]
background = np.empty_like(rgb)
background[..., 0] = 0.08
background[..., 1] = 0.26
background[..., 2] = 0.48
composite = rgb * alpha_2d[..., None] + background * (1.0 - alpha_2d[..., None])

fig, axes = plt.subplots(2, 2, figsize=(16, 10), constrained_layout=True)
axes[0, 0].imshow(rgb)
axes[0, 0].set_title('Input RGB (DiffMatte retriever)')
axes[0, 1].imshow(tri, cmap='gray', vmin=0.0, vmax=1.0)
axes[0, 1].set_title('Trimap: background / unknown / foreground')
axes[1, 0].imshow(alpha_2d, cmap='gray', vmin=0.0, vmax=1.0)
axes[1, 0].set_title('Predicted alpha (CUDAExecutionProvider)')
axes[1, 1].imshow(composite)
axes[1, 1].set_title('Alpha composite on blue background')
for axis in axes.flat:
    axis.axis('off')

visualization_path = ARTIFACT_DIR / 'retriever_gpu_visualization.png'
fig.savefig(visualization_path, dpi=160, bbox_inches='tight')
plt.show()
print(visualization_path)

In [ ]:
report = {
    'model': str(MODEL_PATH),
    'model_sha256': actual_hash,
    'image': str(IMAGE_PATH),
    'trimap': str(TRIMAP_PATH),
    'input_shape': list(image.shape),
    'seed': 0,
    'providers': providers,
    'elapsed_seconds': elapsed_seconds,
    'alpha_png': str(alpha_path),
    'visualization_png': str(visualization_path),
}
report_path = ARTIFACT_DIR / 'retriever_gpu_report.json'
report_path.write_text(json.dumps(report, indent=2) + '\n', encoding='utf-8')
report